<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 40px 30px; border-radius: 12px; margin-bottom: 24px;">
  <h1 style="color: #ffffff; font-size: 2.2em; margin: 0;">📊 Lab 05: Evaluate Your Advisory Agent</h1>
  <p style="color: #a0cfee; font-size: 1.15em; margin-top: 8px;">Zava Bank Workshop — Microsoft Foundry Agent Observability</p>
</div>

<div style="background-color: #1a1a2e; border-left: 4px solid #00b4d8; padding: 20px 24px; border-radius: 8px; margin-bottom: 20px;">

## 🎓 What We Learn

In this lab you will:

- **Run batch evaluations** for quality metrics — coherence, relevance, groundedness, and fluency
- **Run safety evaluations** — violence, sexual content, self-harm, and hate/unfairness
- **Interpret results** in the context of a financial services agent
- **Understand why groundedness is the single most critical metric** for an agent that handles real portfolio data

Evaluation is how you move from _"it seems to work"_ to _"here is the systematic evidence that it works."_

</div>

<div style="background-color: #1a1a2e; border-left: 4px solid #e0a800; padding: 20px 24px; border-radius: 8px; margin-bottom: 20px;">

## 🏦 The Zava Bank Story

The advisory agent looks good in ad-hoc testing — advisors ask a question, the agent responds with plausible portfolio data, and everyone nods.

But the compliance team needs more than anecdotes. They need **systematic evidence**:

- Are responses **accurate** against the underlying portfolio, market, and risk data?
- Does the agent **hallucinate** portfolio values or fabricate market indicators?
- Does the agent stay **grounded in facts**, or does it invent financial advice?
- Is the output **safe** — free of content that could constitute market manipulation or discriminatory advice?

Evaluation at scale answers these questions. It turns subjective confidence into auditable metrics.

</div>

---
## 1 · Setup

In [ ]:
import os, json, time
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
credential = DefaultAzureCredential()

project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"Endpoint : {endpoint}")
print(f"Model    : {model_name}")

---
## 2 · Create the Advisory Agent

Same agent definition used in Lab 02 — a Zava Bank financial advisor with access to client, market, and risk data.

In [ ]:
SYSTEM_PROMPT = """
You are Zava Bank's Client Advisory Agent — a financial assistant built for
registered investment advisors at Zava Bank.

Your responsibilities:
- Provide portfolio summaries, risk analysis, and market context using ONLY the
  data available through your tools and grounding sources.
- Present numbers exactly as they appear in the data. Never estimate, round
  creatively, or fabricate figures.
- When asked about holdings, risk metrics, or market indicators, cite the
  specific data source (client_portfolios, risk_metrics, market_data).
- If a question falls outside available data, say so clearly rather than
  guessing.
- Maintain a professional, concise tone appropriate for financial services.

You must NEVER:
- Provide specific buy/sell/hold recommendations (you are informational only).
- Fabricate portfolio values, risk metrics, or market data.
- Offer tax advice, legal advice, or guarantees of future performance.
"""

agent = project_client.agents.create(
    name="zava-bank-advisor-eval",
    description="Zava Bank advisory agent for evaluation lab",
    agent=PromptAgentDefinition(
        model=model_name,
        instructions=SYSTEM_PROMPT,
    ),
)

print(f"Agent created: {agent.name} (v{agent.version})")

---
## 3 · Load Evaluation Data

The evaluation dataset is a JSONL file with pre-written queries, context strings, expected responses, and ground-truth summaries covering portfolio, risk, and market scenarios.

In [ ]:
import pandas as pd

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "..", "data", "zava-bank")

eval_data = []
with open(os.path.join(DATA_DIR, "evaluation_data.jsonl"), "r") as f:
    for line in f:
        if line.strip():
            eval_data.append(json.loads(line.strip()))

print(f"Loaded {len(eval_data)} evaluation samples")
print(json.dumps(eval_data[0], indent=2))

---
## 4 · Generate Agent Responses

Run each evaluation query through the live agent. This gives us *actual* agent output to evaluate — not canned responses.

In [ ]:
conversation = openai_client.conversations.create()

agent_responses = []
for i, item in enumerate(eval_data):
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input=item["query"],
    )
    agent_responses.append({
        "query": item["query"],
        "context": item.get("context", ""),
        "response": response.output_text,
        "ground_truth": item.get("ground_truth", ""),
    })
    print(f"  [{i+1}/{len(eval_data)}] {item['query'][:60]}...")

print(f"\nCollected {len(agent_responses)} responses.")

---
## 5 · Prepare JSONL for Batch Evaluation

The Foundry Evaluations API accepts JSONL with `query`, `response`, `context`, and `ground_truth` fields.

In [ ]:
# Format as inline JSONL content for the batch evaluation API
jsonl_content = agent_responses

print(f"Prepared {len(jsonl_content)} records for evaluation.")
print("Sample record:")
print(json.dumps(jsonl_content[0], indent=2)[:500])

---
## 6 · Run Quality Evaluation

We evaluate four quality dimensions. In financial services, **groundedness is the most critical metric** — the agent must not hallucinate portfolio values or fabricate market data.

In [ ]:
from azure.ai.projects.models import EvaluatorConfiguration

# Quality evaluators
quality_evaluators = [
    "coherence",     # Does the advisor give logically structured advice?
    "relevance",     # Does the response answer what was actually asked?
    "groundedness",  # Is the agent using real data or hallucinating? (Critical in FSI)
    "fluency",       # Is the language professional and clear?
]

print("Submitting quality evaluation batch...")
print(f"  Evaluators : {quality_evaluators}")
print(f"  Samples    : {len(jsonl_content)}")
print(f"  Model      : {model_name}")

quality_eval = project_client.evaluations.create(
    evaluation={
        "displayName": "Zava Bank Advisor — Quality Evaluation",
        "data": {
            "type": "custom",
            "jsonl_content": jsonl_content,
        },
        "evaluators": {
            evaluator: EvaluatorConfiguration(
                id=evaluator,
            )
            for evaluator in quality_evaluators
        },
    },
)

print(f"\nEvaluation submitted — ID: {quality_eval.id}")
print("Waiting for results (this may take a few minutes)...")

In [ ]:
# Poll for completion
status = None
while status not in ("Completed", "Failed", "Canceled"):
    time.sleep(15)
    quality_result = project_client.evaluations.get(quality_eval.id)
    status = quality_result.status
    print(f"  Status: {status}")

if status == "Failed":
    print(f"Evaluation failed: {quality_result}")
else:
    print("Quality evaluation complete.")

---
## 7 · Display Quality Results

In [ ]:
# Parse per-evaluator metrics from the result
quality_metrics = quality_result.metrics if hasattr(quality_result, "metrics") else {}

print("=" * 60)
print("  QUALITY EVALUATION RESULTS — Zava Bank Advisor")
print("=" * 60)

rows = []
for metric_name, metric_value in quality_metrics.items():
    rows.append({"Metric": metric_name, "Score": metric_value})
    print(f"  {metric_name:<20s}  {metric_value}")

if rows:
    df_quality = pd.DataFrame(rows)
    display(df_quality.style.set_caption("Quality Metrics").format({"Score": "{:.3f}"}))
else:
    print("  (Detailed metrics not yet available — check the Foundry portal for full results.)")
    print(f"  Evaluation ID: {quality_eval.id}")

---
## 8 · Run Safety Evaluation

Financial services agents face unique safety risks — for example, generating content that could constitute market manipulation or discriminatory lending advice. These evaluators check for harmful output across standard safety categories.

In [ ]:
safety_evaluators = [
    "violence",
    "sexual",
    "self_harm",
    "hate_unfairness",
]

print("Submitting safety evaluation batch...")
print(f"  Evaluators : {safety_evaluators}")
print(f"  Samples    : {len(jsonl_content)}")

safety_eval = project_client.evaluations.create(
    evaluation={
        "displayName": "Zava Bank Advisor — Safety Evaluation",
        "data": {
            "type": "custom",
            "jsonl_content": jsonl_content,
        },
        "evaluators": {
            evaluator: EvaluatorConfiguration(
                id=evaluator,
            )
            for evaluator in safety_evaluators
        },
    },
)

print(f"\nSafety evaluation submitted — ID: {safety_eval.id}")
print("Waiting for results...")

In [ ]:
status = None
while status not in ("Completed", "Failed", "Canceled"):
    time.sleep(15)
    safety_result = project_client.evaluations.get(safety_eval.id)
    status = safety_result.status
    print(f"  Status: {status}")

if status == "Failed":
    print(f"Evaluation failed: {safety_result}")
else:
    print("Safety evaluation complete.")

In [ ]:
safety_metrics = safety_result.metrics if hasattr(safety_result, "metrics") else {}

print("=" * 60)
print("  SAFETY EVALUATION RESULTS — Zava Bank Advisor")
print("=" * 60)

rows = []
for metric_name, metric_value in safety_metrics.items():
    rows.append({"Metric": metric_name, "Score": metric_value})
    print(f"  {metric_name:<20s}  {metric_value}")

if rows:
    df_safety = pd.DataFrame(rows)
    display(df_safety.style.set_caption("Safety Metrics").format({"Score": "{:.3f}"}))
else:
    print("  (Detailed metrics not yet available — check the Foundry portal for full results.)")
    print(f"  Evaluation ID: {safety_eval.id}")

---
## 9 · Interpreting the Results

<div style="background-color: #1a1a2e; border-left: 4px solid #00b4d8; padding: 20px 24px; border-radius: 8px;">

### Quality Metrics — What the Scores Mean

| Metric | What It Measures | Why It Matters at Zava Bank |
|---|---|---|
| **Coherence** | Does the advisor give logically structured, internally consistent responses? | Advisors need clear narratives they can relay to clients — not a jumble of disconnected facts. |
| **Relevance** | Does the response actually answer what was asked? | An advisor who asks about Margaret Chen's portfolio should not get a market overview. |
| **Groundedness** | Is the agent using real data, or is it hallucinating? | **The most critical metric in financial services.** A fabricated portfolio value, a made-up risk metric, or an invented market indicator could trigger compliance violations, bad trades, or regulatory action. |
| **Fluency** | Is the language professional and clear? | Client-facing output must be polished. Grammatical errors or awkward phrasing erode trust. |

### Safety Metrics — Lower Is Better

| Metric | Risk in Financial Context |
|---|---|
| **Violence** | Should always be 0 — no plausible scenario in advisory context. |
| **Sexual** | Should always be 0. |
| **Self-harm** | Should always be 0. |
| **Hate / Unfairness** | The meaningful risk. Could the agent generate content that implies discriminatory lending or biased financial advice? Even low-grade bias is a regulatory concern. |

### Grounding: The Metric That Matters Most

In a typical chatbot, a fluent but slightly inaccurate response is a minor issue. In financial services, it is a **material risk**. If the agent says Margaret Chen's portfolio is worth \$155,125 but the real number is \$145,000, an advisor might make a decision based on wrong data. Groundedness evaluation catches this class of error systematically.

</div>

---
## 10 · Cleanup

In [ ]:
project_client.agents.delete(agent_name=agent.name)
print(f"Deleted agent: {agent.name}")
print("Evaluation results remain accessible in the Foundry portal.")

---

<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 30px 28px; border-radius: 12px; margin-top: 24px;">

## ✅ Summary

**Key takeaway** — Evaluation is how you move from _"the agent seems fine"_ to _"here is systematic, auditable evidence that the agent is accurate and compliant."_

What we did:
1. Generated live agent responses for a structured evaluation dataset
2. Ran **quality evaluation** — coherence, relevance, groundedness, fluency
3. Ran **safety evaluation** — violence, sexual, self-harm, hate/unfairness
4. Interpreted the results in a financial services context

In regulated industries, this evidence may be **required for audit purposes**. The Foundry Evaluations API makes it possible to run these checks as part of CI/CD, not just as one-off manual tests.

### ➡️ Next: Lab 06 — Red Teaming

Evaluation tells you how the agent performs on expected queries. Red teaming tells you how it performs when someone is *actively trying to break it*.

</div>